# Perlin Noise — Python Baseline for ludoSpring

**SPDX-License-Identifier: AGPL-3.0-or-later**

Deterministic Perlin 2D/3D and fBm reference values for procedural generation.
The Rust implementation must match these exactly (within floating-point tolerance).

### References
- Perlin, K. (1985). "An image synthesizer." SIGGRAPH '85.
- Perlin, K. (2002). "Improving noise." SIGGRAPH '02.

In [ ]:
import math

# Standard Perlin permutation table (duplicated to 512 for wrapping)
PERM_BASE = [
    151, 160, 137, 91, 90, 15, 131, 13, 201, 95, 96, 53, 194, 233, 7, 225,
    140, 36, 103, 30, 69, 142, 8, 99, 37, 240, 21, 10, 23, 190, 6, 148,
    247, 120, 234, 75, 0, 26, 197, 62, 94, 252, 219, 203, 117, 35, 11, 32,
    57, 177, 33, 88, 237, 149, 56, 87, 174, 20, 125, 136, 171, 168, 68, 175,
    74, 165, 71, 134, 139, 48, 27, 166, 77, 146, 158, 231, 83, 111, 229, 122,
    60, 211, 133, 230, 220, 105, 92, 41, 55, 46, 245, 40, 244, 102, 143, 54,
    65, 25, 63, 161, 1, 216, 80, 73, 209, 76, 132, 187, 208, 89, 18, 169,
    200, 196, 135, 130, 116, 188, 159, 86, 164, 100, 109, 198, 173, 186, 3, 64,
    52, 217, 226, 250, 124, 123, 5, 202, 38, 147, 118, 126, 255, 82, 85, 212,
    207, 206, 59, 227, 47, 16, 58, 17, 182, 189, 28, 42, 223, 183, 170, 213,
    119, 248, 152, 2, 44, 154, 163, 70, 221, 153, 101, 155, 167, 43, 172, 9,
    129, 22, 39, 253, 19, 98, 108, 110, 79, 113, 224, 232, 178, 185, 112, 104,
    218, 246, 97, 228, 251, 34, 242, 193, 238, 210, 144, 12, 191, 179, 162, 241,
    81, 51, 145, 235, 249, 14, 239, 107, 49, 192, 214, 31, 181, 199, 106, 157,
    184, 84, 204, 176, 115, 121, 50, 45, 127, 4, 150, 254, 138, 236, 205, 93,
    222, 114, 67, 29, 24, 72, 243, 141, 128, 195, 78, 66, 215, 61, 156, 180,
]
PERM = PERM_BASE + PERM_BASE  # Duplicate for mod-free indexing

In [ ]:
def fade(t):
    """Perlin improved fade: 6t^5 - 15t^4 + 10t^3"""
    return t * t * t * (t * (t * 6.0 - 15.0) + 10.0)

def lerp(t, a, b):
    return a + t * (b - a)

def grad2d(hash_val, x, y):
    """2D gradient from hash."""
    h = hash_val & 3
    if h == 0: return x + y
    if h == 1: return -x + y
    if h == 2: return x - y
    return -x - y

def perlin2d(x, y):
    """Classic 2D Perlin noise at (x, y)."""
    xi = int(math.floor(x)) & 255
    yi = int(math.floor(y)) & 255
    xf = x - math.floor(x)
    yf = y - math.floor(y)
    u = fade(xf)
    v = fade(yf)
    aa = PERM[PERM[xi] + yi]
    ab = PERM[PERM[xi] + yi + 1]
    ba = PERM[PERM[xi + 1] + yi]
    bb = PERM[PERM[xi + 1] + yi + 1]
    x1 = lerp(u, grad2d(aa, xf, yf), grad2d(ba, xf - 1, yf))
    x2 = lerp(u, grad2d(ab, xf, yf - 1), grad2d(bb, xf - 1, yf - 1))
    return lerp(v, x1, x2)

In [ ]:
# Canonical test points — these golden values are in Rust certification/constants.rs
test_points = [(0.0, 0.0), (0.5, 0.5), (1.0, 1.0), (3.14, 2.72), (10.5, 7.3)]

print(f"{'Point':<16} {'Perlin2D':>15}")
print("-" * 33)
for x, y in test_points:
    val = perlin2d(x, y)
    print(f"({x:>5.2f}, {y:>5.2f})  {val:>15.12f}")

# Origin must be exactly 0
assert perlin2d(0.0, 0.0) == 0.0, "Perlin at origin must be 0"
print("\nPerlin(0,0) = 0: PASS")

In [ ]:
def fbm2d(x, y, octaves=4, lacunarity=2.0, persistence=0.5):
    """Fractal Brownian Motion: sum of Perlin octaves."""
    total = 0.0
    amplitude = 1.0
    frequency = 1.0
    for _ in range(octaves):
        total += perlin2d(x * frequency, y * frequency) * amplitude
        amplitude *= persistence
        frequency *= lacunarity
    return total

# fBm test points
print(f"{'Point':<16} {'fBm (4 octaves)':>16}")
print("-" * 34)
for x, y in [(0.5, 0.5), (3.14, 2.72), (10.5, 7.3)]:
    val = fbm2d(x, y)
    print(f"({x:>5.2f}, {y:>5.2f})  {val:>16.12f}")

## Validation

These values are compiled into `ludoSpring/barracuda/src/certification/constants.rs`
and compared in the `s_procedural_gen` validation scenario. The Rust implementation
of Perlin noise is in `barracuda/src/procedural/noise.rs`.